[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/intensivedatacomp/WignerCamp2026/blob/main/altx/altx_exercises.ipynb)

# `altx` — Adaptive Law-Based Transformation — Exercises

In the lecture you computed `altx` by hand: extract sliding windows, embed them
into small symmetric matrices, take the eigenvector of the smallest absolute
eigenvalue (the **law**), project test windows onto it, and summarise the
result into a feature.

The `altx` package (<https://github.com/halmosb/altx>) does exactly this, but
for real datasets with hundreds of instances and thousands of time points —
by hand that would take forever.

In this notebook each topic is introduced with a worked **example** that
reproduces a calculation from the slides, followed by a short **exercise**.
Fill in the cells marked `# TODO`.

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive/')
    # Your working copy on Google Drive.
    !mkdir -p "/content/drive/My Drive/WignerCamp2026/altx"
    %cd /content/drive/My\ Drive/WignerCamp2026/altx
    # altx is not on PyPI yet, so install straight from GitHub.
    # [examples] also pulls in `aeon`, which we use to load the GunPoint dataset.
    !pip install --quiet "git+https://github.com/halmosb/altx.git#egg=altx[examples]"
except ImportError:
    IN_COLAB = False
    %load_ext autoreload
    %autoreload 2
print(f'Running on {"Google colab" if IN_COLAB else "Local computer"}')

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from altx import ALT as Altx
from altx import ExtractMethods

torch.manual_seed(0)
np.random.seed(0)

## 1. From the slides to code

Recall the toy dataset from the lecture:

- Class **a**: `a = [1, 1, 2, 3, 5]`
- Class **b**: `b = [2, 1, 5, 7, 17]`
- Test instance: `x = [2, 3, 5, 8, 13]`

With `r = 3`, `l = 2`, `k = 1`, you computed by hand that the test instance is
**much closer to class a** (mean squared projection `≈ 0.00035`) than to
class b (`≈ 0.029`).

`altx` uses slightly different parameter names: `r → R`, `l → L`, `k → K`.
Let's reproduce the same numbers with the package.

In [ ]:
# One training instance per class — altx expects shape (N, T):
# N instances, each of length T.
train_data = torch.tensor([
    [1., 1., 2., 3., 5.],   # class a
    [2., 1., 5., 7., 17.],  # class b
])
train_classes = torch.tensor([0, 1])  # 0 = class a, 1 = class b

# R = 3, L = 2, K = 1, exactly like on the slides.
model = Altx(train_data, train_classes, R=3, L=2, K=1)
model.train()
model.print_number_of_laws()

`print_number_of_laws` shows how many laws were extracted **per class**.
Class a has three overlapping windows of length 3 in a 5-element series
(`[1,1,2]`, `[1,2,3]`, `[2,3,5]`) — exactly the three windows from slide 8.

Now transform the test instance with the `mean_all` extraction method, which
is precisely "mean of all squared projection values" — the calculation from
slide 12.

In [ ]:
x = torch.tensor([2., 3., 5., 8., 13.])

features = model.transform(x, [["mean_all"]])
print(f"feature vector: {features}")
print(f"  class a score: {features[0]:.5f}  (slides: 0.00035)")
print(f"  class b score: {features[1]:.5f}  (slides: 0.029)")

The feature vector has one entry **per class** — that's what
`len(RLK) × noc × n_methods × m = 1 × 2 × 1 × 1 = 2` means. The lower score
wins: `x` belongs to class **a**, matching slide 12 exactly (small numerical
differences are just floating-point rounding).

### Exercise 1

Slide 13 asks you to classify a *different* test instance using the same
`a`/`b` training set:

```
x = [3, -1, 5, 3, 13]
```

The slide's answer (computed with the original `altx_intro.ipynb` notebook):
class a score `≈ 0.43`, class b score `≈ 0.42` → class **b** (just barely!).

1. Build the same `model` again (you can reuse the one above — it's already
   trained, so there's no need to retrain).
2. Transform the new test instance with `mean_all`.
3. Print both scores and the predicted class, and check that they roughly
   match the slide.

In [ ]:
# TODO: transform the new test instance and report both class scores.
x_ex1 = torch.tensor([3., -1., 5., 3., 13.])

# features_ex1 = model.transform(...)
# print(...)

## 2. The parameters `R`, `L`, `K`

From the slides:

- **`R`**: length of the window cut out of the original series.
- **`L`**: embedding size — the window is folded into an `L × L` matrix.
- **`K`**: step (stride) between consecutive windows.

`altx` adds one constraint that wasn't spelled out as a formula in the
slides, but follows directly from how the embedding works:

$$ (R - 1) \bmod (2L - 2) = 0 $$

In other words, `R` must be expressible as `R = s · (2L - 2) + 1` for some
positive integer `s`. If you don't pick a valid `R`, the package raises a
`ValueError` rather than silently doing something wrong — that's exactly the
kind of "fail loudly" behaviour you want from a numerical library.

If you don't give `R` at all, `altx` defaults to the smallest valid window,
`R = 2L - 1`, which is the `2l - 1` you saw on slide 16.

In [ ]:
for L_try in (2, 3, 4):
    valid_R = [s * (2 * L_try - 2) + 1 for s in range(1, 4)]
    print(f"L = {L_try}: smallest valid R values are {valid_R}")

In [ ]:
# A deliberately *invalid* (R, L) pair to see altx complain.
bad_data = torch.randn(4, 50)
bad_classes = torch.tensor([0, 0, 1, 1])

try:
    Altx(bad_data, bad_classes, R=6, L=3, K=1)  # (6-1) % (2*3-2) = 5 % 4 != 0
except ValueError as err:
    print(f"altx raised ValueError: {err}")

### Exercise 2

1. For `L = 5`, work out **by hand** (pen and paper, or a one-line
   calculation) the three smallest valid values of `R`.
2. Build an `Altx` model with `L = 5` and your *smallest* valid `R`, using
   `K = 2`, on `torch.randn(4, 60)` / `torch.tensor([0, 0, 1, 1])`. Confirm it
   does **not** raise.
3. Then try `R = R_valid + 1` (one more than your valid value) and confirm
   it **does** raise a `ValueError`. Wrap it in a `try`/`except` like the
   example above so the notebook doesn't stop.

In [ ]:
# TODO 1: valid R values for L = 5 (write them as a comment or a list)
# valid_R_L5 = [...]

# TODO 2: build a model with L=5, your smallest valid R, K=2 — should succeed.
# model_ex2 = Altx(torch.randn(4, 60), torch.tensor([0, 0, 1, 1]), R=..., L=5, K=2)

# TODO 3: build a model with R = (your valid R) + 1 — should raise ValueError.
# try:
#     Altx(...)
# except ValueError as err:
#     print(f"altx raised ValueError: {err}")

## 3. A real dataset: Gun Point

Time to leave the five-element toy series behind. We'll use the **Gun Point**
dataset from slide 4 — the same one shown in the lecture. It comes from the
UCR Time Series Classification Archive and is bundled with the `aeon`
package, which `altx` itself uses in its own examples.

Each series records the `x`-coordinate of an actor's hand. In half the
recordings the actor draws a real gun from a hip holster, points it, and
returns it (class `Gun`); in the other half they perform the same motion
with an empty hand (class `Point`). The two classes look almost identical
except for a small timing difference — exactly the kind of subtle pattern
`altx`'s laws are designed to pick up.

In [ ]:
from aeon.datasets import load_classification

X, y = load_classification("GunPoint")
y = y.astype(np.int8)  # labels arrive as strings ("1"/"2"); altx wants integers

print(f"X shape: {X.shape}   (n_instances, n_channels, n_timepoints)")
print(f"y shape: {y.shape}, classes present: {np.unique(y)}")

# altx accepts (N, T) for univariate data, so drop the singleton channel axis.
X = X[:, 0, :]

# A simple, reproducible train/test split (the real archive split is closer
# to 50/150, but a 50/50 split trains faster for a live exercise).
rng = np.random.default_rng(0)
n = X.shape[0]
idx = rng.permutation(n)
split = n // 2
train_idx, test_idx = idx[:split], idx[split:]

train_data = torch.tensor(X[train_idx], dtype=torch.float32)
train_classes = torch.tensor(y[train_idx] - 1, dtype=torch.long)  # 1,2 -> 0,1
test_data = torch.tensor(X[test_idx], dtype=torch.float32)
test_classes = torch.tensor(y[test_idx] - 1, dtype=torch.long)

print(f"train: {train_data.shape}, test: {test_data.shape}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for cls, name, color in [(0, "Gun", "crimson"), (1, "Point", "steelblue")]:
    sample = train_data[train_classes == cls][0]
    ax.plot(sample, label=name, color=color)
ax.set_xlabel("t (a. u.)")
ax.set_ylabel("y (a. u.)")
ax.set_title("Gun Point — one example per class")
ax.legend()
plt.tight_layout()
plt.show()

Now train `altx` on the training split and transform the test split.
We'll use `L = 4` (so the default `R = 2·4 - 1 = 7`) and `K = 1`.

In [ ]:
model_gp = Altx(train_data, train_classes, L=4, K=1)
print(f"RLK triplets: {model_gp.RLK}")
model_gp.train()
model_gp.print_number_of_laws()

features_train = model_gp.transform_set(train_data, [["mean_all"]])
features_test = model_gp.transform_set(test_data, [["mean_all"]])
print(f"feature shape (train): {features_train.shape}")
print(f"feature shape (test):  {features_test.shape}")

Each instance now has **two** features: its mean squared projection
onto the "Gun" laws and onto the "Point" laws. Classification is the same
rule as on the slides — predict whichever class has the **smaller** score.

In [ ]:
def predict_by_min_score(features):
    """Predict the class with the smallest score (column index)."""
    return torch.argmin(features, dim=1)

predictions = predict_by_min_score(features_test)
accuracy = (predictions == test_classes).float().mean().item()
print(f"Test accuracy with mean_all, L=4, K=1: {accuracy:.3f}")

### Exercise 3

1. Train a **new** model (`model_ex3`) on the same `train_data` /
   `train_classes`, but with `L = 3` and `K = 2` instead.
2. Transform `test_data` with `mean_all` and compute the test accuracy the
   same way as above.
3. In a markdown cell or comment, compare the two accuracies and the two
   models' `print_number_of_laws()` output. Which configuration found more
   laws? Did more laws translate into better accuracy here?

In [ ]:
# TODO 1: build and train model_ex3 with L=3, K=2 on the GunPoint training data.
# model_ex3 = Altx(...)
# model_ex3.train()
# model_ex3.print_number_of_laws()

# TODO 2: transform test_data and compute accuracy.
# features_ex3 = model_ex3.transform_set(test_data, [["mean_all"]])
# predictions_ex3 = predict_by_min_score(features_ex3)
# accuracy_ex3 = (predictions_ex3 == test_classes).float().mean().item()
# print(f"Test accuracy with mean_all, L=3, K=2: {accuracy_ex3:.3f}")

# TODO 3: your comparison here, as a comment.

## 4. Extraction methods

`mean_all` — the mean of *every* squared projection — is the simplest
extraction method, and the one used throughout the slides. `altx` ships a
few others that summarise the projection scores differently:

| method | what it computes |
| --- | --- |
| `mean_all` | mean of all squared scores (no percentile) |
| `mean` | mean of scores near a chosen percentile `q` |
| `var` | variance of scores near percentile `q` |
| `excess_kurtosis` | how "spiky" vs. "flat" the score distribution is |
| `nth_moment` (e.g. `2nd_moment`) | the `n`-th central moment |

You can pass **several** methods at once as a list of `[method, percentile]`
pairs (the percentile is omitted for `mean_all`). Each one multiplies the
length of the resulting feature vector.

In [ ]:
extr_methods = [["mean_all"], ["mean", 0.05], ["var", 0.1]]

features_multi = model_gp.transform(test_data[0], extr_methods)
print(f"feature shape: {features_multi.shape}")
print(f"  = len(RLK) x noc x n_methods x m = "
      f"{len(model_gp.RLK)} x {model_gp.noc} x {len(extr_methods)} x {model_gp.m}")
print(features_multi)

The 6 numbers are ordered `(class, method)`: first all 3 methods for
class 0 (Gun), then all 3 for class 1 (Point) — see the "How the feature
vector is structured" section of the `altx` docs if you want the full
general rule for multiple `(R, L, K)` triplets and multiple channels.

### Exercise 4

1. Use `model_gp.transform_set` to compute features for the **whole** test
   set with `extr_methods = [["mean_all"], ["2nd_moment", 0.1]]`.
2. Confirm the resulting shape matches
   `len(RLK) × noc × n_methods × m`.
3. Build a classifier that, for each test instance, sums the two `mean_all`
   columns... no wait — instead, build a classifier that uses **only the
   `2nd_moment` columns** (ignore `mean_all` this time) and predicts the
   class with the smaller score, exactly like before. Compute its accuracy
   and compare it to the `mean_all`-only accuracy from Section 3.

In [ ]:
# TODO 1 & 2: compute features_ex4 and check the shape.
# features_ex4 = model_gp.transform_set(test_data, [["mean_all"], ["2nd_moment", 0.1]])
# print(features_ex4.shape)

# TODO 3: slice out the 2nd_moment columns (hint: columns are ordered
# (class 0, mean_all), (class 0, 2nd_moment), (class 1, mean_all), (class 1, 2nd_moment))
# second_moment_cols = features_ex4[:, [1, 3]]
# predictions_ex4 = predict_by_min_score(second_moment_cols)
# accuracy_ex4 = (predictions_ex4 == test_classes).float().mean().item()
# print(f"Test accuracy with 2nd_moment only: {accuracy_ex4:.3f}")

## 5. Anomaly detection

Slides 22–29 treat anomaly detection as classification with a **single**
class: train laws on normal data only, then look at *where in the test
series* the projection score spikes, instead of summarising the whole
series into one number per class.

The current version of `altx` doesn't have a ready-made one-line anomaly
detector (`plot_anomalies` is a placeholder for future work — try calling it
and you'll see it isn't implemented yet). But every other piece you've
learned is enough to reproduce the slides' anomaly-detection calculation by
hand, the same way you reproduced the classification calculation in
Section 1.

The key building block is `model.multiply_only(z, rlk)`: it embeds `z` and
projects every window onto the stored laws — stopping *before* the
classification summary step, which is exactly what we want for a
per-window anomaly score.

In [ ]:
# Normal data: a perfect Fibonacci-like sequence, just like slide 23.
normal = torch.tensor([[1., 1., 2., 3., 5., 8., 13.]])
normal_classes = torch.tensor([0])  # a single class: "normal"

model_anomaly = Altx(normal, normal_classes, R=3, L=2, K=1)
model_anomaly.train()
model_anomaly.print_number_of_laws()

In [ ]:
# Test instance with anomalies at positions 0 and 3 — identical to slide 24.
x_anom = torch.tensor([6., 1., 2., 8., 5., 8., 13.])

rlk = model_anomaly.RLK[0]
F = model_anomaly.multiply_only(x_anom, rlk)
print(f"projection shape: {F.shape}")  # (n_windows, n_laws)

# Slide 26: average the squared projection across laws, one score per window.
scores = (F ** 2).mean(dim=1)
print(f"scores: {scores.numpy().round(3)}")
print("(slides: [0.567, 0.008, 0.092, 0.196, 0.000, 0.000])")

In [ ]:
tau = 0.15  # same threshold as slide 26
anomaly_positions = torch.nonzero(scores > tau).flatten()
print(f"anomalies detected at positions: {anomaly_positions.tolist()}")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(scores.numpy(), "o", label="score")
ax.axhline(tau, ls="--", color="black", label="threshold")
ax.set_xlabel("window index")
ax.set_ylabel("anomaly score")
ax.legend()
plt.tight_layout()
plt.show()

This matches slide 26 exactly: peaks at window indices **0** and **3**.

Remember the index-mapping rule from slides 27–29: the score vector has
`N - L + 1` entries (here `7 - 2 + 1 = 6` ✓), and a peak at score-index `j`
flags an anomaly starting at original position `j`.

### Exercise 5

**Part A — index mapping (no code needed).** A time series has length
`N = 12` and `altx` is run with `L = 4`.

1. How many entries does the anomaly score vector have?
2. If the score vector has a peak at index `j = 5`, which position in the
   original series does that correspond to?

Write your answers as a comment in the code cell below, then check Part A(1)
with a one-line calculation.

**Part B — a new signal.** Build your own "normal" training series of at
least 8 points that follows some simple rule of your choosing (constant
differences, doubling, alternating, anything goes), train an `altx` model on
it with `R = 3, L = 2, K = 1` exactly like above, then construct a test
series that follows your rule everywhere **except** at two positions you
choose. Compute the per-window scores with `multiply_only`, pick a
reasonable threshold `tau`, and check whether the detected peaks land at the
positions where you broke the rule.

In [ ]:
# Part A
# TODO: N - L + 1 for N=12, L=4, as a comment, then verified:
N_ex5, L_ex5 = 12, 4
# n_scores_ex5 = ...
# print(n_scores_ex5)

# TODO: anomaly position for a peak at score-index j=5 (as a comment)

# Part B
# TODO: build your own rule-following series and inject two anomalies.
# my_normal = torch.tensor([[...]])
# model_ex5 = Altx(my_normal, torch.tensor([0]), R=3, L=2, K=1)
# model_ex5.train()

# my_test = torch.tensor([...])
# rlk_ex5 = model_ex5.RLK[0]
# F_ex5 = model_ex5.multiply_only(my_test, rlk_ex5)
# scores_ex5 = (F_ex5 ** 2).mean(dim=1)
# print(scores_ex5)

# TODO: pick tau_ex5 and print the detected anomaly positions.

## Summary

- `Altx(train_set, train_classes, R, L, K, device).train()` extracts one
  *law* (eigenvector of smallest absolute eigenvalue) per sliding window of
  every training instance, grouped by class.
- `R`, `L`, `K` are the slides' `r`, `l`, `k`. They must satisfy
  `(R - 1) % (2L - 2) == 0`; if you omit `R`, the default `2L - 1` is always
  valid. Passing an invalid `R` raises `ValueError` immediately.
- `model.transform(z, extr_methods)` / `model.transform_set(...)` project a
  test instance (or a whole set) onto the stored laws and summarise the
  result into a feature vector of length
  `len(RLK) × noc × n_methods × m`. Classify by picking the class with the
  **smallest** score.
- Extraction methods (`mean_all`, `mean`, `var`, `excess_kurtosis`,
  `nth_moment`) control *how* the per-window projection scores are
  summarised; you can combine several at once.
- Anomaly detection is classification with one class: train on normal data
  only, then look at the **per-window** scores (via `multiply_only`, since
  `plot_anomalies` isn't implemented yet) instead of summarising the whole
  series. A score above a threshold `tau` flags an anomaly at that window's
  starting position.